In [ ]:
import pandas as pd 
import sqlite3
import unidecode
from difflib import SequenceMatcher
from typing import Callable, List, Sequence

def select_dynamic_versions(
    versions: Sequence[str],
    target_change: float = 0.20,
    min_step: int = 1,
    change_fn: Callable[[str, str], float] | None = None,
) -> List[int]:
    """
      Return version indices (1-based) like [1, t1, t2, ...] such that each jump
      is as close as possible to target_change.

      Greedy rule:
        - start at i
        - scan j = i+min_step, i+min_step+1, ...
        - if change(i,j) first crosses target, pick j or j-1 (whichever is closer)
        - if never crosses, pick the farthest j (the last version)
        - set i = chosen j and repeat

      Args:
        versions: list of version texts in order (v1..vn)
        target_change: desired change magnitude (e.g., 0.20)
        min_step: minimum forward jump in index
        change_fn: optional custom diff function in [0,1].
                   default is 1 - SequenceMatcher.ratio().

      Returns:
        1-based indices into `versions`.
    """
    if not versions:
        return []
    if len(versions) == 1:
        return [1]
    if not (0 < target_change < 1):
        raise ValueError("target_change must be in (0, 1).")
    if min_step < 1:
        raise ValueError("min_step must be >= 1.")

    def default_change(a: str, b: str) -> float:
        return 1.0 - SequenceMatcher(None, a or "", b or "").ratio()

    diff = change_fn or default_change
    n = len(versions)

    selected = [1]  # 1-based
    i = 0  # 0-based pointer

    while i < n - 1:
        start_j = i + min_step
        if start_j >= n:
            break

        prev_j = start_j
        prev_change = diff(versions[i], versions[prev_j])

        # If first candidate already crosses target, choose it.
        if prev_change >= target_change:
            chosen_j = prev_j
        else:
            chosen_j = None
            for j in range(start_j + 1, n):
                cur_change = diff(versions[i], versions[j])
                if cur_change >= target_change:
                    # choose between prev_j and j by closeness to target
                    if abs(prev_change - target_change) <= abs(cur_change - target_change):
                        chosen_j = prev_j
                    else:
                        chosen_j = j
                    break
                prev_j, prev_change = j, cur_change

            if chosen_j is None:
                # never reached target; take farthest available
                chosen_j = n - 1

        if chosen_j <= i:
            break  # safety

        selected.append(chosen_j + 1)  # back to 1-based
        i = chosen_j

    return selected

In [6]:
con = sqlite3.connect('../data/google-docs/google_docs_versions.db')

In [7]:
pd.read_sql('''SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;''',  con)

,name
0,entryversion


In [10]:
rfi_df = pd.read_sql('''SELECT * from entryversion''',  con)

In [11]:
ls ../data/google-docs/fetch_google_docs.py

__pycache__/             google_docs_versions.db
fetch_google_docs.py     revisions/


In [14]:
rfi_df['summary'].isnull().value_counts()

summary
False    229
True       8
Name: count, dtype: int64

In [30]:
rfi_df['summary'] = rfi_df['summary'].apply(lambda x: unidecode.unidecode(x) if x != None else '')

In [33]:
(rfi_df['summary'] == '').value_counts()

summary
False    225
True      12
Name: count, dtype: int64

In [35]:
rfi_df['entry_id'].unique()

array([ 524314818,  140033046, 1395714255,  183330940,  253268898,
       1455753760,  131686781, 2088360662])

In [44]:
rfi_df.to_csv('../data/google-docs/rfi_df.csv')

# Parse

In [122]:
import os
from openai import OpenAI
os.environ['OPENAI_API_KEY'] = open('/Users/spangher/.openai-salt-lab-key.txt').read().strip()

client = OpenAI()
def prompt_openai(prompt, tier='flex'):
    # tier= flex, priority, default
    resp = client.responses.create(
        model="gpt-5",
        input=prompt,
        service_tier=tier
    )
    return resp.output_text

In [110]:
# variables: article_v1, article_v2
diff_prompt = open('../news-edits-pipeline/prompts/ALEX_edit_actions_pair.prompt').read()
diff_prompt = diff_prompt.replace('{', '{{').replace('}', '}}').replace('{{{{', '{').replace('}}}}', '}')

In [ ]:
rfi_df = rfi_df.loc[lambda df: df['summary'] != '']
one_rfi = rfi_df.loc[lambda df: df['entry_id'] == 524314818]
subsampled_versions = select_dynamic_versions(one_rfi['summary'].tolist())
subsampled_versions = list(map(lambda x: x -1, subsampled_versions))

In [90]:
subsampled_rfi = one_rfi.iloc[subsampled_versions]

In [95]:
version_pairs = list(zip(subsampled_versions[:-1], subsampled_versions[1:]))

In [130]:
idx = 2
summary_1, summary_2 = one_rfi.iloc[list(version_pairs[idx])]['summary']

In [131]:
input_prompt = diff_prompt.format(article_v1=summary_1, article_v2=summary_2)

In [132]:
response_2 = prompt_openai(input_prompt, tier='priority')

In [126]:
response_1 = response

In [134]:
from util import highlight_pair, format_article_html

In [136]:
left_html, right_html = highlight_pair(summary_1, summary_2)

In [137]:
format_article_html(524314818, version_pairs[idx][0], version_pairs[idx][1], left_html, right_html)

In [146]:
from IPython.display import display, Markdown
import json

In [151]:
for action_row in json.loads(response_2):
    # display(Markdown(f"### Edit action {action_idx + 1}"))
    display(Markdown(f"**Editor request**\n\n{action_row['request'] or ''}"))
    display(Markdown(f"**Writer action**\n\n{action_row['writer action'] or ''}"))
    display(Markdown(f"**Content added**\n\n{action_row['content added'] or ''}"))
    display(Markdown(f"**Content removed**\n\n{action_row['content removed'] or ''}"))
    display(Markdown(f"**Content changed**\n\n{action_row['content changed'] or ''}"))

display(Markdown("---"))

**Editor request**

Cut the long historical background and front-load a clear, persuasive lede that explains the AI inflection point and the need for a new, team-science research model focused on the public good.

**Writer action**

Removed the internet/ARPANET history section and drafted new opening paragraphs under the 'AI Presents an Inflection Point' section to establish the thesis and policy frame. Workshopped the lede with policy colleagues to ensure the argument aligns with OSTP’s RFI goals.

**Content added**

AI Presents an Inflection Point

We are at a unique inflection point in history where AI holds the potential to unlock unprecedented opportunities across healthcare, education, scientific discovery, and every sector of society. However, realizing this transformative potential requires a fundamental reimagining of how we conduct research. New collaborative research models built on team science can leverage the great potential breakthroughs in academia with the talent, resources, and scale of private sector research, all while focusing on scientific discovery for the public good. 

**Content removed**

Background


Breakthrough technologies that have defined modern life were developed through a partnership that academia and government forged together. Perhaps no better example of this is the invention of the internet itself. Thanks to the government's investments in university research centers through the Advanced Projects Research Agency (ARPA) institutions such as Stanford were well-poised in the 1960s to engage in curiosity-driven computer research. When the Advanced Research Projects Agency Network (ARPANET) was established, it was a rudimentary system of networks with extremely high barriers to entry. However, as academic researchers continued to innovate and improve the network it was able to grow into the internet where potential commercial applications started to reveal themselves. 


It wasn't until the late 1980s that the commercial opportunities associated with the internet began to reveal themselves. And it wasn't until the 1990s that home internet service began to really allow consumers to interact with the internet directly. Through the multiple decades of the internet's growth, academic research centers fueled the continued growth of the technology to the point where the internet could be fully embedded into consumer products and thus the daily lives of people around the world. 

**Content changed**



...

**Writer action**

Added a sentence listing emblematic academic breakthroughs (deep learning, ImageNet, diffusion models) and kept the investment/model-share statistics for context.

**Content added**

While foundational AI research -- advancements such as deep learning, ImageNet, and diffusion models -- was born in academic research labs, the majority of today's frontier research into AI is taking place in the private sector. Private investment in AI in the United States hit $193.96 billion in 2025.[1] Industry produced 90% of the most notable AI models compared with 60% the previous year.[2]

**Content removed**

AI[a] has turned this model for scientific and technological research on its head because the majority frontier research into AI is not taking place in academic labs but in the private sector. 

**Content changed**



---

## Clean API text

In [65]:
"""
I am trying to parse documents to understand the text added by writers vs. the outline that was already given.

Here is text given by the writer. Just return the writer-added content, nothing else. 
If everything is writer-added content and you don't detect outlines anywhere, then just return "Done".

<text>
{text}
</text>


"""

'\nI am trying to parse documents to understand the text added by writers vs. the outline that was already given.\n\nHere is text given by the writer:\n\n{text}\n'

In [53]:
idx = 3
print(one_rfi['summary'].iloc[idx])

Call
AGENCY:
Office of Science and Technology Policy.
ACTION:
Request for information.
SUMMARY:
The Office of Science and Technology Policy (OSTP) requests input from all interested parties on Federal policy updates that aim to accelerate the American scientific enterprise, enable groundbreaking discoveries, and ensure that scientific progress and technological innovation benefit all Americans. Through this Request for Information (RFI), OSTP seeks input from academia; private sector organizations; industry groups; state, local, and tribal governments; and other stakeholders regarding priorities for strengthening the science and technology (S&T) ecosystem to support both the expansion of scientific knowledge and the mechanisms to transition these discoveries into the marketplace. This RFI will inform the formulation of Executive branch efforts to advance and maintain U.S. S&T leadership.
DATES:
Interested persons are invited to submit comments on or before 11:59 p.m. (ET) December 26, 